In [1]:
from manim import *
from scipy.integrate import solve_ivp

c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [2]:
class GlowDot(Dot3D):
  def __init__(self, point=ORIGIN, color=WHITE, radius=0.1, glow_radius=0.3, layers=15, **kwargs):
    kwargs.pop('renderer', None)
    kwargs.pop('scene', None)
    
    super().__init__(radius=radius, color=color, **kwargs)

    self.glow = VGroup()
    for i in range(layers):
      glow_sphere = Sphere(
        radius=radius + i * glow_radius / layers,
        color=color,
        resolution=(10, 10)
      ).move_to(point)
      opacity = 0.4 * (1 - i / layers) ** 3  # Cubic falloff
      glow_sphere.set_opacity(opacity)
      glow_sphere.set_stroke(width=0) 
      self.glow.add(glow_sphere)

    self.add_to_back(self.glow)
    self.move_to(point)

  def set_glow_color(self, color):
    self.set_color(color)
    self.glow.set_color(color)
    return self

  def move_to(self, point):
    super().move_to(point)
    self.glow.move_to(point)
    return self

In [3]:
class TestGlowDot(ThreeDScene):
  def construct(self):
    FRAME_WIDTH = config.frame_width
    FRAME_HEIGHT = config.frame_height

    axes = ThreeDAxes(
      x_range=(-50, 50, 5),
      y_range=(-50, 50, 5),
      z_range=(0, 50, 5),
      x_length=16,
      y_length=16,
      z_length=8
    )
    axes.set_width(FRAME_WIDTH)
    axes.set_height(FRAME_HEIGHT)
    axes.center()
    self.set_camera_orientation(phi=80*DEGREES, theta=135*DEGREES, zoom=1.7)
    dot = GlowDot(color=RED, radius=0.01, glow_radius=0.5)
    self.add(axes, dot)
    self.play(dot.animate.move_to(UP * 2 + RIGHT * 3 + SMALL_BUFF))
    # self.play(Rotate(dot, angle=2*PI, about_point=ORIGIN), run_time=5)
    self.wait()

In [25]:
class TestLatex(ThreeDScene):
  def construct(self):
    # Equations
    equations = MathTex(
            R"""
            \begin{aligned}
            \frac{\mathrm{d} x}{\mathrm{~d} t} & =\sigma(y-x) \\
            \frac{\mathrm{d} y}{\mathrm{~d} t} & =x(\rho-z)-y \\
            \frac{\mathrm{d} z}{\mathrm{~d} t} & =x y-\beta z
            \end{aligned}
            """,
            color=WHITE,
            font_size=25
          )
    # Fix to top left of the screen with additional padding
    equations.to_corner(UL)
    equations.shift(0.5 * RIGHT + 0.5 * DOWN)
    self.add_fixed_in_frame_mobjects(equations)
    self.add(equations)
    self.play(Write(equations), run_time = 2)
    

In [29]:
def lorenz_system(t, state, sigma=10, rho=28, beta=8/3):
  x, y, z = state
  dxdt = sigma * (y-x)
  dydt = x * (rho - z) - y
  dzdt = x * y - beta * z
  return [dxdt, dydt, dzdt]

def ode_solution_points(function, initial, time, dt=0.01):
  solution = solve_ivp(
    function,
    t_span=(0, time),
    y0=initial,
    t_eval=np.arange(0, time, dt)
  )
  return solution.y.T

class LorenzAttractor(ThreeDScene):
  def construct(self):
    x_range = (-20, 20, 5)
    y_range = (-25, 25, 5)
    z_range = (0, 50, 5)
    # Set up the axes
    axes = ThreeDAxes(
      x_range=x_range,
      y_range=y_range,
      z_range=z_range,
      x_length=6,   # Proportional lengths keep the shape correct
      y_length=6,
      z_length=6,
      axis_config={"include_tip": False}
    )
    axes.move_to(ORIGIN)
    self.set_camera_orientation(phi=80*DEGREES, theta=120*DEGREES)
    self.add(axes)
  
    # Equations
    equations = MathTex(
            R"""
            \begin{aligned}
            \frac{\mathrm{d} x}{\mathrm{~d} t} & =\sigma(y-x) \\
            \frac{\mathrm{d} y}{\mathrm{~d} t} & =x(\rho-z)-y \\
            \frac{\mathrm{d} z}{\mathrm{~d} t} & =x y-\beta z
            \end{aligned}
            """,
            color=WHITE,
            font_size=25
          )
    # Fix to top left of the screen with additional padding
    equations.to_corner(UL)
    equations.shift(0.5 * RIGHT + 0.5 * DOWN)
    self.add_fixed_in_frame_mobjects(equations)
    self.add(equations)
    self.play(Write(equations), run_time = 2)
    

    # Display Lorenz solutions
    epsilon = 1e-3 # Small change in each curve
    # Time it takes for the evolution of all the lorenz attractors
    evolution_time = 15
    # How many curves we want to animate
    num_of_curves = range(5) 

    # All of the initial states the function will start at
    states = [
      [10, 10, 10 + n * epsilon] for n in num_of_curves
    ]
    
    colors = color_gradient([BLUE_E, BLUE_A], len(states))
    
    # Create all the curves
    curves = VGroup()
    for state, color in zip(states, colors):
      # Get all solution points with set IVPs
      points = ode_solution_points(lorenz_system, state, evolution_time)
      # The curve vector object.
      curve = VMobject().set_points_as_corners(axes.c2p(points))
      # Sets color (gradient or solid) and line thickness
      curve.set_stroke(color=color, width=1.5)
      # Add to group    
      curves.add(curve)

    # A dots group filled with dots of specific radius and glow radius for each color in the gradient
    dots = VGroup(GlowDot(color=color, radius=0.01, glow_radius=0.2) for color in colors)

    # Function to update all the dots at the end of their respective curve
    def update_dots(dots):
      for dot, curve in zip(dots, curves):
        if curve.get_num_points() > 0:
          dot.move_to(curve.get_end())
    
    # Add the updater to dots so it can actually update
    dots.add_updater(update_dots)

    # Add dots to the scene and play
    self.add(dots)
    self.begin_ambient_camera_rotation(rate=3 * DEGREES)
    self.play(*(
      Create(curve, rate_func = linear) for curve in curves), # Draws each curve in group curves
      run_time = evolution_time
    )
    self.stop_ambient_camera_rotation()
    self.wait()
    


In [30]:
manim -pqh LorenzAttractor

Manim Community v0.20.0

[02/28/26 04:13:40] INFO     Animation 0 : Partial movie file written in                   ]8;id=157011;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=893964;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py#590\590]8;;\
                             'C:\Users\dragi\Documents\manimations\code\media\videos\code\                         
                             1080p60\partial_movie_files\LorenzAttractor\3986606113_737837                         
                             244_2419370985.mp4'                                                                   

[02/28/26 04:41:46] INFO     Animation 1 : Partial movie file written in                   ]8;id=67562;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=940728;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py#590\590]8;;\
                             'C:\Users\dragi\Documents\manimations\code\media\videos\code\                         
                             1080p60\partial_movie_files\LorenzAttractor\944021846_3075001                         
                             459_3492442828.mp4'                                                                   

[02/28/26 04:41:53] INFO     Animation 2 : Partial movie file written in                   ]8;id=627134;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=624473;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py#590\590]8;;\
                             'C:\Users\dragi\Documents\manimations\code\media\videos\code\                         
                             1080p60\partial_movie_files\LorenzAttractor\4164205629_351553                         
                             3520_2049391617.mp4'                                                                  

                    INFO     Combining to Movie file.                                      ]8;id=123950;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=626506;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py#742\742]8;;\

                    INFO                                                                   ]8;id=344993;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=563793;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py#893\893]8;;\
                             File ready at                                                                         
                             'C:\Users\dragi\Documents\manimations\code\media\videos\code\                         
                             1080p60\LorenzAttractor.mp4'                                                          
                                                                                                                   

                    INFO     Rendered LorenzAttractor                                                  ]8;id=373130;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene.py\scene.py]8;;\:]8;id=409188;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene.py#278\278]8;;\
                             Played 3 animations                                                                   

[02/28/26 04:41:54] INFO     Previewed File at:                                                     ]8;id=986102;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\utils\file_ops.py\file_ops.py]8;;\:]8;id=962961;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\utils\file_ops.py#236\236]8;;\
                             'C:\Users\dragi\Documents\manimations\code\media\videos\code\1080p60\L                
                             orenzAttractor.mp4'                                                                   